# PSEi thesis figure gallery

Run all to display the revised numbered gallery and regenerate plots without training. The current gallery replaces Figures 1.2 and 1.3 with reproducible loss illustrations on an observed pre-test PSEi return. Candidate forecasts are hypothetical. The current DOCX-extracted images are in `figures/paper/`; original synthetic Figures 1.2 and 1.3 remain in `figures/legacy-paper/`.

See `docs/figure-alignment-2026-10-28.md` for live Google Docs status and the remaining diagram updates. The DOCX is the 28 October 2026 export of the live PSEi revision.


In [ ]:
import os, subprocess, sys, tempfile
from pathlib import Path

try:
    import google.colab
    in_colab = True
except ImportError:
    in_colab = False

repo = Path.cwd()
if in_colab:
    # A fresh checkout prevents an older Colab session showing stale figures.
    repo = Path(tempfile.mkdtemp(prefix='psei-figures-', dir='/content')) / 'stock-prediction'
    subprocess.run(['git', 'clone', '--branch', 'codex/with-sop-2', 'https://github.com/marianoluiz/stock-prediction.git', str(repo)], check=True)
elif not (repo / 'scripts' / 'build_paper_figures.py').is_file():
    raise RuntimeError('Run this notebook from the research repository root or Google Colab.')
os.chdir(repo)
assert (repo / 'docs' / 'with-sop-2.docx').is_file()
print('Repository:', repo)
print('Git revision:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('Current gallery: figures/paper; paper: docs/with-sop-2.docx')


## Build the gallery
The script checks the committed input CSV hashes, recreates the Chapter 1 loss illustrations and retained legacy examples, copies the paper's Chapter 3 diagrams, and redraws Chapter 4 from saved result data. In Colab, install `yfinance` because the repository's plotting modules import the shared data module; this run never downloads prices. Colab already supplies PyTorch, NumPy, pandas, Matplotlib, and SciPy.


In [ ]:
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yfinance'], check=True)
subprocess.run([sys.executable, 'scripts/build_paper_figures.py'], check=True)


## Revised gallery, in paper order
These are the revised gallery assets; see the alignment audit for live-document status. The regenerated versions are in `figures/generated/` with the same numbered names; `manifest.json` records hashes and source categories.


In [ ]:
import json
from IPython.display import Image, Markdown, display
manifest = json.loads((repo / 'figures/generated/manifest.json').read_text(encoding='utf-8'))
for item in manifest['figures']:
    display(Markdown(f"### Figure {item['number']} — {item['status']}"))
    display(Image(filename=str(repo / item['current']), width=760))
print('All', len(manifest['figures']), 'paper figures displayed.')


## Optional: download both sets as a ZIP
Run this cell only if you want a local download. It includes exact paper assets, regenerated versions, and the manifest.


In [ ]:
import shutil
zip_path = Path('/content/PSEi_Thesis_Figures.zip') if 'google.colab' in sys.modules else repo / 'PSEi_Thesis_Figures.zip'
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', root_dir=repo, base_dir='figures')
print(zip_path)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(str(zip_path))
